<a href="https://colab.research.google.com/github/praveenambati888-lang/movie-review-sentiment/blob/main/movie_review_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets

In [4]:
import pandas as pd
from datasets import load_dataset

data = load_dataset("imdb")
train = pd.DataFrame(data["train"])
test = pd.DataFrame(data["test"])

print(train.shape, test.shape)
print(train["label"].value_counts())
print(train["text"][0][:300])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

(25000, 2) (25000, 2)
label
0    12500
1    12500
Name: count, dtype: int64
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = TfidfVectorizer(stop_words="english", max_features=20000)
X_train = vec.fit_transform(train["text"])
X_test = vec.transform(test["text"])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, train["label"])

preds = model.predict(X_test)
print(classification_report(test["label"], preds))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [6]:
from sklearn.metrics import accuracy_score

train["clean"] = train["text"].str.replace("<br />", " ", regex=False)
test["clean"] = test["text"].str.replace("<br />", " ", regex=False)

vec2 = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train2 = vec2.fit_transform(train["clean"])
X_test2 = vec2.transform(test["clean"])

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, train["label"])
print("Accuracy:", accuracy_score(test["label"], model2.predict(X_test2)))

Accuracy: 0.89452


In [7]:
words = vec2.get_feature_names_out()
coef = pd.Series(model2.coef_[0], index=words).sort_values()
print("Most NEGATIVE:")
print(coef.head(8))
print("\nMost POSITIVE:")
print(coef.tail(8))

def predict_review(text):
    prob = model2.predict_proba(vec2.transform([text]))[0][1]
    return ("Positive" if prob > 0.5 else "Negative"), round(prob, 2)

print(predict_review("This movie was not good at all"))
print(predict_review("An absolute masterpiece, I loved every minute"))

Most NEGATIVE:
bad         -7.830152
worst       -7.241518
awful       -5.866945
the worst   -5.764128
boring      -5.519431
poor        -5.094855
waste       -4.943541
no          -4.444603
dtype: float64

Most POSITIVE:
fun          3.558649
amazing      3.702696
the best     3.758284
best         3.800192
perfect      4.308583
wonderful    4.415574
excellent    5.371001
great        7.137697
dtype: float64
('Negative', np.float64(0.09))
('Positive', np.float64(0.81))
